# Practical 4B — CIFAR-10 CNN Architecture Comparison

## Aim
Implement and compare LeNet, AlexNet-style, VGG-style and ResNet50 architectures on CIFAR-10.

## Setup
Run the installation cell if required. These notebooks use Python 3.10–3.12 where possible. On Apple Silicon, TensorFlow/PyTorch acceleration depends on the installed build; CPU execution is supported.

**Important:** dataset downloads and model training may take time. Results are generated when you run the notebook; no accuracy values are pre-filled.

In [ ]:
%pip install -q numpy pandas matplotlib scikit-learn tensorflow

## Reproducibility\nA fixed random seed is used where practical. If a dataset download fails, check your internet connection and rerun the loading cell.

## 1. Load CIFAR-10 and prepare labels

In [ ]:
import time, numpy as np, pandas as pd, matplotlib.pyplot as plt, tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import classification_report
np.random.seed(42); tf.random.set_seed(42)
(x_train,y_train),(x_test,y_test)=keras.datasets.cifar10.load_data()
x_train=x_train.astype("float32")/255.; x_test=x_test.astype("float32")/255.
y_train=y_train.ravel(); y_test=y_test.ravel()

## 2. Define LeNet, AlexNet-style, VGG16 and ResNet50\nFor CIFAR-10's 32×32 images, AlexNet and VGG-style models are adapted to smaller input dimensions. ResNet50 uses a 32×32 input and random initialization (not ImageNet weights).

In [ ]:
def make_arch(name):
    inp=keras.Input((32,32,3)); x=inp
    if name=="LeNet":
        x=layers.Conv2D(6,5,activation="relu",padding="same")(x); x=layers.AveragePooling2D()(x)
        x=layers.Conv2D(16,5,activation="relu")(x); x=layers.AveragePooling2D()(x)
        x=layers.Flatten()(x); x=layers.Dense(120,activation="relu")(x); x=layers.Dense(84,activation="relu")(x)
    elif name=="AlexNet":
        x=layers.Conv2D(64,3,activation="relu",padding="same")(x); x=layers.MaxPooling2D()(x)
        x=layers.Conv2D(128,3,activation="relu",padding="same")(x); x=layers.MaxPooling2D()(x)
        x=layers.Conv2D(256,3,activation="relu",padding="same")(x); x=layers.GlobalAveragePooling2D()(x)
        x=layers.Dense(256,activation="relu")(x); x=layers.Dropout(.5)(x)
    elif name=="VGG16":
        for filters in [64,128,256]:
            for _ in range(2): x=layers.Conv2D(filters,3,padding="same",activation="relu")(x)
            x=layers.MaxPooling2D()(x)
        x=layers.GlobalAveragePooling2D()(x)
    elif name=="ResNet50":
        # Keras ResNet50 requires 32x32 or larger; include_top=False supports this input.
        x=keras.applications.ResNet50(include_top=False,weights=None,input_tensor=inp)(inp)
        x=layers.GlobalAveragePooling2D()(x)
    out=layers.Dense(10,activation="softmax")(x)
    m=keras.Model(inp,out); m.compile(optimizer="adam",loss="sparse_categorical_crossentropy",metrics=["accuracy"])
    return m

## 3. Train and compare\nA reduced subset and short run are used for practical feasibility. ResNet50 is computationally expensive; reduce the subset further if needed.

In [ ]:
results=[]
for name in ["LeNet","AlexNet","VGG16","ResNet50"]:
    model=make_arch(name); start=time.time()
    model.fit(x_train[:10000],y_train[:10000],epochs=2,batch_size=128,validation_split=.1,verbose=1)
    loss,acc=model.evaluate(x_test[:2000],y_test[:2000],verbose=0)
    results.append((name,acc,loss,time.time()-start))
    print(name,acc,loss)
display(pd.DataFrame(results,columns=["Architecture","Accuracy (2k test subset)","Loss","Training time (s)"]))

## 4. Conclusion\nCompare architecture depth, parameter count, accuracy and training time. These are simplified/adapted implementations, not exact canonical AlexNet/VGG16 training recipes. ResNet50 here is trained from scratch.